# 12. CloudXeus Orders API — Azure Function App (Key-Secured Auth)

This is the **security-hardened iteration** of `11. cloudxeus-func/function_app.py`. The two files are
otherwise byte-for-byte identical (`diff` confirms it) except for **one change, applied twice**:

```diff
-           auth_level=func.AuthLevel.ANONYMOUS)
+           auth_level=func.AuthLevel.FUNCTION)
```

Both routes moved from `ANONYMOUS` (no auth) to `FUNCTION` (a function/host key required on every request).
The accompanying `cloudxeus_spec.json` changed to match: it now includes a `"security"` section declaring a
`functionKeyHeader` API-key scheme (`x-functions-key` header), so the OpenAPI document itself documents that
callers must present a key — meaning an Azure AI Foundry agent configured with this API as an OpenAPI tool
would need that key supplied as part of its tool's authentication configuration (e.g. a connection storing the
key), not just a bare unauthenticated endpoint URL.

**Why this change matters:** `11.`'s anonymous version is convenient for local development, but exposing an
anonymous HTTP endpoint on the public internet (once deployed to a real Function App URL) is a real production
security gap — anyone who finds the URL can call it. Requiring a function key is the minimal fix that still
keeps the API simple to consume (a single static key, no full OAuth flow) while ensuring only holders of that
key — including a properly configured Foundry OpenAPI tool — can call it.

**Difficulty:** Intermediate


## Prerequisites

Same as `11. cloudxeus-func/function_app.ipynb`:

**Pip packages:**
- `azure-functions` — not in root `requirements.txt`, install with `pip install azure-functions` (also listed
  in this folder's own `requirements.txt`).

**Other tooling required:**
- Azure Functions Core Tools (`func` CLI) to run locally.
- A storage emulator (Azurite) — `local.settings.json` still sets `"AzureWebJobsStorage": "UseDevelopmentStorage=true"`.

**Azure resource(s) required for a real deployment:** An Azure Function App resource + Storage Account.

**Configuration:** Still governed by **`local.settings.json`** locally / Application Settings once deployed —
not this repo's root `.env`. The function **key itself** (needed to call `FUNCTION`-level routes) is generated
by the Functions runtime — locally, `func start` prints a default/host key at startup, or you can fetch a
specific function's key via `func azure functionapp list-functions <app-name> --show-keys` once deployed, or
from the Azure Portal's "Function Keys" blade.


## What You'll Learn

- The practical difference `auth_level=func.AuthLevel.FUNCTION` makes vs. `ANONYMOUS`: every request now needs
  a valid function key
- How to actually supply that key when testing locally with `curl`
- How an OpenAPI spec (`cloudxeus_spec.json`) documents an API-key auth requirement via a `securitySchemes`
  entry and a top-level `security` array
- Why hardening a Function App's `auth_level` matters specifically when that Function App is meant to be
  called by an Azure AI Foundry agent as a custom tool


### How to actually run and test this locally

Same `func start` workflow as the `11.` variant, but now every request must include the function key:

```bash
cd "14-azure-ai103/02. Section Code/12. cloudxeus-func1"
pip install -r requirements.txt
func start
# func start's console output includes a default host key you can use for local testing
```

Test with the key supplied via the `x-functions-key` header (matching `cloudxeus_spec.json`'s documented
`functionKeyHeader` scheme):

```bash
curl -H "x-functions-key: <your-local-or-host-key>" http://localhost:7071/api/orders
curl -H "x-functions-key: <your-local-or-host-key>" http://localhost:7071/api/orders/1001
```

Omitting the header (or supplying a wrong key) against a `FUNCTION`-level route returns an HTTP `401
Unauthorized` — the key visibly matters here, unlike the `11.` variant.

- **💡 Exam tip:** AI-102 expects you to know Azure Functions' key-based auth options (`FUNCTION` keys,
  scoped to one function; `HOST`/master keys, scoped to the whole app) as one authentication mechanism
  available when a Function App is consumed by another Azure AI service — alongside Entra ID/managed identity
  auth, which is generally the stronger recommended option for service-to-service calls where supported.
- **🔄 Alternatives:** Rather than a static function key, a production-grade version of this API might use
  **Azure AD (Entra ID) authentication** on the Function App itself (App Service authentication / "Easy Auth")
  or front it with **API Management** for finer-grained auth, rate limiting, and key rotation — function keys
  are the simplest option, not the strongest.


### App initialization and sample data (unchanged from `11.`)

Identical to the anonymous variant: `func.FunctionApp()` plus the same in-memory `ORDERS` dictionary. No
secrets live in this code — the key requirement is enforced by the Functions **runtime**, based on each
route's declared `auth_level`, not by any code inside the function body itself.

- **💡 Exam tip:** This is worth noting explicitly: **auth enforcement here happens outside your function
  code**, at the Functions host/runtime level, based on the `auth_level` you declare in the `@app.route(...)`
  decorator. Your function body never has to check a key manually.
- **🔄 Alternatives:** N/A for this cell — this part is identical to `11. cloudxeus-func`.


In [ ]:
import azure.functions as func
import json

app = func.FunctionApp()

ORDERS = {
    "1001": {
        "order_id": "1001",
        "customer": "Northgate Retail",
        "status": "shipped",
        "total": 2450.00,
    },
    "1002": {
        "order_id": "1002",
        "customer": "Aurora Logistics",
        "status": "processing",
        "total": 815.50,
    },
    "1003": {
        "order_id": "1003",
        "customer": "Brightline Media",
        "status": "delivered",
        "total": 129.99,
    },
}


### `GET /orders` — now requires a function key

Identical body to `11.`'s `list_orders`, but `auth_level=func.AuthLevel.FUNCTION` means the Functions runtime
now rejects any request that doesn't present a valid key (via `x-functions-key` header or `?code=` query
string) **before** this function body even runs.

- **💡 Exam tip:** Because key validation happens at the runtime/host level, a request with a missing/invalid
  key never reaches your Python code at all — you'll see a `401` from the Functions host itself, not from
  anything `list_orders` returns.
- **🔄 Alternatives:** `?code=<key>` as a query-string parameter is the alternative to the `x-functions-key`
  header for supplying the key — functionally equivalent, but a header is generally preferred since query
  strings are more likely to leak into logs/browser history.


In [ ]:
@app.route(route="orders", methods=["GET"],
           auth_level=func.AuthLevel.FUNCTION)
def list_orders(req: func.HttpRequest) -> func.HttpResponse:
    return func.HttpResponse(
        json.dumps(list(ORDERS.values())),
        mimetype="application/json",
    )


### `GET /orders/{order_id}` — now requires a function key

Same route-parameter and 404-handling logic as the `11.` variant's `get_order`, again now gated by
`auth_level=func.AuthLevel.FUNCTION`. Together with `cloudxeus_spec.json`'s new `security` block, this is the
complete picture: the OpenAPI spec *describes* the auth requirement (for any tool/consumer that reads the
spec, including a Foundry agent's OpenAPI tool configuration), and `auth_level=FUNCTION` is what actually
*enforces* it at runtime — the two must be kept in sync, since nothing automatically derives one from the
other.

- **💡 Exam tip:** When registering an external REST API as an OpenAPI tool for an Azure AI Foundry agent, the
  tool configuration needs to know *how* to authenticate to that API (anonymous / API key / connection /
  managed identity) — this is exactly the kind of `securitySchemes` metadata `cloudxeus_spec.json` now
  carries, and it's what would let you supply the function key securely (e.g. via a stored connection) rather
  than embedding it in a prompt or plaintext config.
- **🔄 Alternatives:** N/A — this cell mirrors `get_order` from `11.` with only the `auth_level` changed.


In [ ]:
@app.route(route="orders/{order_id}", methods=["GET"],
           auth_level=func.AuthLevel.FUNCTION)
def get_order(req: func.HttpRequest) -> func.HttpResponse:
    order_id = req.route_params.get("order_id")
    order = ORDERS.get(order_id)

    if order is None:
        return func.HttpResponse(
            json.dumps({"error": f"Order {order_id} not found"}),
            status_code=404,
            mimetype="application/json",
        )

    return func.HttpResponse(
        json.dumps(order),
        mimetype="application/json",
    )


## Summary

This notebook is functionally identical to `11. cloudxeus-func/function_app.ipynb` except for one security
upgrade applied to both routes: `auth_level` moved from `ANONYMOUS` to `FUNCTION`, and `cloudxeus_spec.json`
was updated to document the resulting `x-functions-key` requirement via an OpenAPI `securitySchemes` entry.
Run locally with `func start` and tested with `curl -H "x-functions-key: ..."`, the API's behavior is
otherwise the same two endpoints as before — the difference only shows up when you *omit* the key and get a
`401` instead of a normal response.


## Try It Yourself

1. **Easy:** Run `func start` for this version, then try the same `curl` commands from the `11.` notebook
   *without* the `x-functions-key` header — confirm you get a `401` instead of the order data.
2. **Intermediate:** Retrieve your local host key from the `func start` console output (or via `func azure
   functionapp list-functions --show-keys` once deployed) and successfully call both endpoints with it.
3. **Advanced:** Compare `cloudxeus_spec.json` between `11.` and `12.` field-by-field (the `security` array
   and `components.securitySchemes.functionKeyHeader` object) and sketch what an Azure AI Foundry OpenAPI
   tool's authentication configuration would need to look like to call this `12.` version successfully,
   versus the trivial "no auth config needed" case for the `11.` version.
